### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

from imports import *

load_dotenv()

In [4]:
## step1 : Load and split the dataset
loader = TextLoader("langchain-crewai-dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)


In [ ]:
chunks

In [6]:
### step 2: Vector Store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

## step 3:MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3414.37it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001ED7BEA61D0>, search_type='mmr', search_kwargs={'k': 5})

In [7]:
## step 4 : LLM and Prompt

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

llm=init_chat_model("openai:o4-mini")
llm


ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'o4-mini', 'release_date': '2025-04-16', 'last_updated': '2025-04-16', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 100000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001ED7BEA5210>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001ED03115290>, root_client=<openai.OpenAI object at 

In [8]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'o4-mini', 'release_date': '2025-04-16', 'last_updated': '2025-04-16', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 100000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': T

In [9]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'LangChain memory     \n– context retention, conversation state management, long-term/short-term memory in LangChain  \n– memory modules: BufferMemory, ConversationBufferMemory, ConversationSummaryMemory, CombinedMemory  \n– vector-store memory & embedding stores (VectorStoreMemory, FAISS, Chroma, Pinecone, Weaviate)  \n– persistent memory backends: SQL-Memory, Redis, MongoDB, DynamoDB  \n– retrieval-augmented generation (RAG) and memory orchestration  \n– session memory vs. rolling/summary memory strategies  \n– memory keying, pruning, serialization, cache invalidation  \n– integration patterns: Agents with memory, Tool use + memory, chain-of-thought state preservation  \n– use cases: chatbots, personalized assistants, multi-turn dialogue, LLM-powered workflows'

In [12]:
# RAG answering prompt
# from langchain_core.prompts import PromptTemplate

answer_prompt = PromptTemplate.from_template("""
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.

If the answer is not present in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
""")

In [14]:
# Step 5: Full RAG pipeline with query expansion
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def retrieve_context(x):
    expanded_query = query_expansion_chain.invoke({"query": x["input"]})

    print("Expanded Query:", expanded_query)

    docs = retriever.invoke(expanded_query)

    print(f"Retrieved {len(docs)} documents")

    return format_docs(docs)


rag_pipeline = (
    RunnableMap(
        {
            "question": lambda x: x["input"],
            "context": retrieve_context,
        }
    )
    | answer_prompt
    | llm
    | StrOutputParser()
)

In [16]:
# Step 6: Run query
response = rag_pipeline.invoke(
    {
        "input": "What is LangChain?"
    }
)

print(response)

Expanded Query: Expanded query:  
“What is LangChain – an open-source framework or library/SDK for building applications with large language models (LLMs)? Include definitions, core features and modules (prompt templates, chains, agents, memory, retrievers, document loaders, vector stores, embeddings), key concepts (chain-of-thought, retrieval-augmented generation/RAG, prompt engineering, agent-based workflows, multi-step reasoning), supported languages (Python, JavaScript/TypeScript), integrations (OpenAI, Hugging Face, Cohere, Anthropic, Pinecone, Weaviate, Milvus, Elasticsearch, Supabase, SQL databases), typical use cases (chatbots, question answering, summarization, code generation, virtual assistants), architecture and design patterns, performance benchmarks, tutorials and examples, best practices, and comparisons with similar tools (LlamaIndex/GPT Index, Haystack).”
Retrieved 5 documents
LangChain is an open-source framework for developing applications powered by large language m

In [17]:
# Step 6: Run query
response = rag_pipeline.invoke(
    {
        "input": "CrewAI Agents?"
    }
)

print(response)

Expanded Query: Expanded query (include synonyms, technical terms, context):

("CrewAI agents" OR "Crew AI agents" OR "CrewAI autonomous agents" OR "CrewAI intelligent agents" OR "CrewAI digital workforce" OR "CrewAI virtual assistants" OR "CrewAI bot framework" OR "CrewAI multi-agent system" OR "CrewAI AI-driven task automation")  
AND  
("platform" OR "framework" OR "architecture" OR "API" OR "SDK" OR "integration" OR "deployment" OR "orchestration" OR "agent management" OR "agent lifecycle")  
AND  
("documentation" OR "developer guide" OR "tutorial" OR "best practices" OR "reference" OR "pricing" OR "use cases" OR "case study" OR "feature list" OR "performance benchmarks")  
AND  
("LLM-based agents" OR "large language model agents" OR "GPT-powered bots" OR "reinforcement learning" OR "chain-of-thought planning" OR "agent coordination" OR "task planning" OR "workflow automation")  
AND  
(comparison OR "vs. AutoGPT" OR "vs. LangChain" OR "vs. Semantic Kernel" OR "alternatives")
Ret